# Python foundations — beginner → advanced (AI engineering track)

**How to use this notebook**

1. Run cells **top to bottom** (each section builds on earlier ideas).
2. Read the markdown **Explanation** blocks before running **Examples**.
3. Modify values and break things — that is how they stick.

4. Try **Part 4 exercises**, then peek **Part 5 solutions** only if needed.

**Requires:** Python 3.10+ recommended (uses `|` union syntax in places). Standard library only except optional comments.

---


## Part 1 — Beginner core

### Variables, types, and operators

**Explanation:** Python figures out types at runtime (`dynamic typing`). You still think in types: numbers for math, strings for text, booleans for conditions. AI pipelines almost always move between **strings** (prompts), **structured dicts/list** (JSON-like tool payloads), and **numbers** (embeddings).

**Examples:**

In [ ]:
# Numbers and strings
batch_size = 8
model_name = "gpt-demo"
temperature = 0.7

# f-strings — you'll use these constantly for prompts
user_prompt = "summarize"
full_prompt = f"Task: {user_prompt}\nBatch size: {batch_size}"
print(full_prompt)

# Booleans and comparisons
use_streaming = True
print(use_streaming and batch_size > 0)

### Lists, tuples, dicts, sets

**Explanation:**
- **list** — ordered, mutable — chunks of text, batches of IDs.
- **tuple** — ordered, immutable — often small fixed records `(role, content)`.
- **dict** — key → value — JSON-shaped API payloads; extremely common.
- **set** — unique elements — dedupe URLs / doc IDs quickly.

**Examples:**

In [ ]:
messages = [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "Hello!"},
]

# Safe access patterns
first_user_text = messages[-1]["content"] if messages else ""
print("Last message:", first_user_text)

# Dict merge (Python 3.9+)
defaults = {"temperature": 0.7, "max_tokens": 256}
overrides = {"temperature": 0.2}
params = defaults | overrides
print(params)

# Unique chunk ids
chunk_ids = ["a", "b", "a", "c"]
print(set(chunk_ids))

### Control flow — `if`, loops

**Explanation:** Branch on errors, empty retrieval results, flags from configs. **`enumerate`** gives index + item — handy when chunking lists.

**Examples:**

In [ ]:
chunks = ["alpha", "beta", "gamma"]

for i, text in enumerate(chunks):
    if len(text) < 4:
        print(i, "short chunk", text)
    else:
        print(i, "ok", text)

### Functions — arguments and returns

**Explanation:** `*args` collects extra positional arguments; `**kwargs` collects keyword arguments. APIs often forward unknown keys → `kwargs` pattern appears in SDK wrappers.

**Examples:**

In [ ]:
def build_messages(system: str, user: str, **extra_fields: str) -> list[dict[str, str]]:
    """Tiny messages builder — extra_fields might hold optional metadata strings."""
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    if extra_fields:
        msgs.append({"role": "system", "content": repr(extra_fields)})
    return msgs


print(build_messages("Be concise.", "Hi", locale="en"))

---
## Part 2 — Intermediate patterns

### Classes and instances

**Explanation:** Bundle state + behavior — tool clients, embedders, chunkers. Prefer **explicit constructors** (`__init__`) over globals.

**Examples:**

In [ ]:
class SimpleEmbedder:
    """Dummy embedder — replaces API with fixed-width fake vectors."""

    def __init__(self, dim: int = 4) -> None:
        self.dim = dim

    def embed(self, texts: list[str]) -> list[list[float]]:
        return [[float(len(t) % self.dim)] * self.dim for t in texts]


emb = SimpleEmbedder(dim=3)
vectors = emb.embed(["hello", "world"])
print(vectors)

### Exceptions — failures are normal in AI systems

**Explanation:** Network timeouts, rate limits, malformed JSON from models — handle **narrowly** (specific exception types where possible), log context, optionally retry at a higher layer.

**Examples:**

In [ ]:
import json


def parse_tool_json(raw: str) -> dict:
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"invalid tool JSON: {e}") from e


print(parse_tool_json('{"tool": "search", "query": "rag"}'))

try:
    parse_tool_json("not json")
except ValueError as err:
    print("Caught:", err)

### Comprehensions — concise transforms

**Explanation:** Same as loops but often clearer for filtering/mapping lists — preprocessing texts before embedding.

**Examples:**

In [ ]:
raw_docs = [" Hello ", "", "WORLD", "  rag  "]

normalized = [s.strip().lower() for s in raw_docs if s.strip()]
print(normalized)

### Files & JSON — ingestion starter kit

**Explanation:** RAG starts with reading sources — JSONL is common (`one JSON object per line`).

**Examples:**

In [ ]:
import json
from pathlib import Path

demo_dir = Path("_notebook_tmp")
demo_dir.mkdir(exist_ok=True)
sample_file = demo_dir / "sample.jsonl"

sample_file.write_text(
    '{"id": 1, "text": "hello"}\n{"id": 2, "text": "world"}\n',
    encoding="utf-8",
)

records = []
for line in sample_file.read_text(encoding="utf-8").splitlines():
    records.append(json.loads(line))

print(records)

# cleanup — optional
sample_file.unlink(missing_ok=True)
demo_dir.rmdir()

---
## Part 3 — Advanced (high value for AI engineers)

### Type hints — contracts without ceremony

**Explanation:** Annotate public functions so teammates + IDEs catch mistakes early. AI codebases glue many libraries — types reduce chaos.

**Examples:**

In [ ]:
from typing import Optional


def cosine_similarity_naive(a: list[float], b: list[float]) -> float:
    if len(a) != len(b):
        raise ValueError("dimension mismatch")
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(y * y for y in b) ** 0.5
    return dot / (na * nb) if na and nb else 0.0


def nearest_neighbor(
    query: list[float],
    corpus: list[list[float]],
    labels: Optional[list[str]] = None,
) -> tuple[int, float]:
    best_i, best_sim = max(
        ((i, cosine_similarity_naive(query, vec)) for i, vec in enumerate(corpus)),
        key=lambda t: t[1],
    )
    return best_i, best_sim


q = [1.0, 0.0]
corpus = [[0.9, 0.1], [0.0, 1.0]]
idx, sim = nearest_neighbor(q, corpus)
print("best idx", idx, "similarity", round(sim, 4))

### `dataclass` — configs and immutable-ish records

**Explanation:** Less boilerplate than manual `__init__`. Use **`frozen=True`** for immutable configs; **`slots=True`** saves memory at scale.

**Examples:**

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True, slots=True)
class RetrievalConfig:
    top_k: int = 5
    score_threshold: float = 0.2


cfg = RetrievalConfig(top_k=3)
print(cfg)

# Frozen prevents accidental mutation — mirrors "good API discipline"
try:
    cfg.top_k = 99
except Exception as e:
    print("Expected error:", type(e).__name__)

### Decorators — cross-cutting behavior

**Explanation:** Wrap functions to add timing, retries, logging — keep core logic clean.

**Examples:**

In [ ]:
import time
import functools


def timed(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        dt = time.perf_counter() - t0
        print(f"{fn.__name__} took {dt*1000:.2f} ms")
        return out

    return wrapper


@timed
def fake_embedding_batch(texts: list[str]) -> list[list[float]]:
    time.sleep(0.05)
    return [[float(len(t))] for t in texts]


fake_embedding_batch(["hello", "rag"])

### Generators — memory-friendly pipelines

**Explanation:** **Yield** items one-by-one instead of building giant lists — critical when iterating millions of chunks.

**Examples:**

In [ ]:
from typing import Iterator


def overlapping_windows(text: str, window: int, step: int) -> Iterator[str]:
    """Simple sliding windows — naive but illustrates generators."""
    i = 0
    while i + window <= len(text):
        yield text[i : i + window]
        i += step


text = "abcdefghij"
print(list(overlapping_windows(text, window=4, step=3)))

### Context managers — predictable cleanup

**Explanation:** Acquire/release resources (files, locks, timers) safely — FastAPI dependency injection echoes this idea.

**Examples:**

In [ ]:
import time
from contextlib import contextmanager


@contextmanager
def timer_label(label: str):
    t0 = time.perf_counter()
    yield
    print(f"[{label}] {1000*(time.perf_counter()-t0):.2f} ms")


with timer_label("fake_retrieval"):
    time.sleep(0.03)

### `Protocol` — structural interfaces

**Explanation:** Anything with matching methods satisfies the protocol — ideal for `Embedder`, `VectorStore`, `Tool` abstractions.

**Examples:**

In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Embedder(Protocol):
    def embed(self, texts: list[str]) -> list[list[float]]: ...


class MiniEmbedder:
    def embed(self, texts: list[str]) -> list[list[float]]:
        return [[1.0, 0.0] for _ in texts]


client: Embedder = MiniEmbedder()
print(isinstance(client, Embedder))
print(client.embed(["hi"]))

### Async basics — concurrency without threads

**Explanation:** Many AI workloads are **I/O bound** (HTTP to models/databases). `asyncio` lets you overlap waits — foundation before FastAPI async routes.

**Examples:** (`asyncio.run` works from scripts and notebooks in modern Python)

In [ ]:
import asyncio


async def fake_remote_call(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return f"result-{name}"


async def main():
    # Run multiple I/O-ish tasks concurrently
    results = await asyncio.gather(
        fake_remote_call("a", 0.2),
        fake_remote_call("b", 0.2),
        fake_remote_call("c", 0.2),
    )
    print(results)


await main()

> **Notebook note:** If `await main()` errors (older Jupyter / IPython), replace the last line with:
> ```python
> asyncio.run(main())
> ```
> Modern Jupyter often supports top-level `await`.

---
## Part 4 — Practice exercises

**How to use**

1. Complete each **Your turn** code cell **before** opening Part 5 (Solutions).
2. Run the cell until **assertions** pass or printed checks succeed.
3. If stuck >15 minutes, peek one hint — then retry without copying.

Difficulty: ★ beginner · ★★ intermediate · ★★★ stretch.


### Exercise 1.1 ★ — Chat-style message list

Build `merge_messages(system_prompt, user_turns)` so it returns a **new** list:
one `{"role":"system","content":...}` followed by each user turn as `{"role":"user","content":...}`.

**Constraints:** Do not mutate `user_turns`.


In [ ]:
def merge_messages(system_prompt: str, user_turns: list[str]) -> list[dict[str, str]]:
    """Return messages: system first, then each user string as its own user message."""
    # YOUR CODE HERE
    raise NotImplementedError


msgs = merge_messages("You are concise.", ["Hi", "Summarize yesterday"])
assert msgs[0] == {"role": "system", "content": "You are concise."}
assert msgs[1] == {"role": "user", "content": "Hi"}
assert msgs[2] == {"role": "user", "content": "Summarize yesterday"}
assert len(msgs) == 3
print("ex 1.1 OK")


### Exercise 1.2 ★ — Filter retrieval scores

Given `doc_scores` as `(doc_id, score)` pairs, keep **ids only** where `score >= threshold`, preserving order.

Use a **list comprehension** (no explicit `append` loop).


In [ ]:
doc_scores = [("doc-a", 0.91), ("doc-b", 0.4), ("doc-c", 0.72)]
threshold = 0.7

# YOUR CODE HERE — replace [] with comprehension
filtered_ids: list[str] = []

assert filtered_ids == ["doc-a", "doc-c"]
print("ex 1.2 OK")


### Exercise 1.3 ★★ — Prompt template with sorted constraints

Implement `prompt_with_constraints(task, constraints)` returning a **single string**:

- First line exactly: `TASK: <task>`
- Then one line per constraint: `<key>: <value>`
- Lines after TASK sorted by **constraint key** alphabetically.


In [ ]:
def prompt_with_constraints(task: str, constraints: dict[str, str]) -> str:
    # YOUR CODE HERE
    raise NotImplementedError


out = prompt_with_constraints(
    "Rewrite email",
    {"tone": "formal", "length": "short"},
)
assert out.startswith("TASK: Rewrite email
")
assert "length: short" in out and "tone: formal" in out
assert out.index("length") < out.index("tone")  # alphabetical keys → length before tone
print("ex 1.3 OK")


---
### Exercise 2.1 ★★ — `TextChunker` class

Implement `chunk(text)` returning contiguous slices each with length **≤ `max_chars`** (characters). No overlapping.


In [ ]:
class TextChunker:
    def __init__(self, max_chars: int) -> None:
        self.max_chars = max_chars

    def chunk(self, text: str) -> list[str]:
        # YOUR CODE HERE
        raise NotImplementedError


tc = TextChunker(5)
assert tc.chunk("abcdefghij") == ["abcde", "fghij"]
assert tc.chunk("hi") == ["hi"]
assert tc.chunk("") == []
print("ex 2.1 OK")


### Exercise 2.2 ★★ — JSON without crashing LLM parsers

Implement `safe_parse_json(raw)` → parsed dict or **`None`** if invalid JSON.

Must **never raise**.


In [ ]:
import json
from typing import Any


def safe_parse_json(raw: str) -> dict[str, Any] | None:
    # YOUR CODE HERE
    raise NotImplementedError


assert safe_parse_json("{}") == {}
assert safe_parse_json("not json") is None
print("ex 2.2 OK")


### Exercise 2.3 ★ — Count JSONL records

`count_jsonl_records(path)` counts lines that successfully parse as JSON.
Skip empty lines.


In [ ]:
from pathlib import Path
import json


def count_jsonl_records(path: Path) -> int:
    # YOUR CODE HERE
    raise NotImplementedError


_tmp = Path("_notebook_tmp_ex")
_tmp.mkdir(exist_ok=True)
_p = _tmp / "sample.jsonl"
_p.write_text('{}

{"x": 1}
', encoding="utf-8")
assert count_jsonl_records(_p) == 2
_p.unlink(missing_ok=True)
_tmp.rmdir()
print("ex 2.3 OK")


---
### Exercise 3.1 ★★ — Typed dot product

Implement `sparse_dot(a, b)` using `zip` + `sum`. Raise `ValueError` if lengths differ.


In [ ]:
from collections.abc import Iterable


def sparse_dot(a: Iterable[float], b: Iterable[float]) -> float:
    # YOUR CODE HERE
    raise NotImplementedError


assert sparse_dot([1, 2, 3], [0.5, 1, -1]) == 1 * 0.5 + 2 * 1 + 3 * (-1)
try:
    sparse_dot([1], [1, 2])
except ValueError:
    print("ex 3.1 OK")
else:
    raise AssertionError("expected ValueError")


### Exercise 3.2 ★★★ — Retry decorator

`@retry_on_value_error(max_attempts=n)` retries the wrapped call when **`ValueError`** is raised,
up to **`n` attempts total** (first try + retries). Other exceptions propagate immediately.

If all attempts fail, re-raise the last `ValueError`.


In [ ]:
import functools


def retry_on_value_error(max_attempts: int = 2):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            # YOUR CODE HERE — retry call on ValueError (max_attempts tries total), else re-raise last
            raise NotImplementedError

        return wrapper

    return decorator


_attempts = {"n": 0}


@retry_on_value_error(max_attempts=3)
def flaky_success():
    _attempts["n"] += 1
    if _attempts["n"] < 3:
        raise ValueError("warmup")
    return "done"


_attempts["n"] = 0
assert flaky_success() == "done"

_attempts["n"] = 0


@retry_on_value_error(max_attempts=2)
def flaky_fail():
    raise ValueError("always")


try:
    flaky_fail()
except ValueError:
    print("ex 3.2 OK")
else:
    raise AssertionError("expected ValueError")


### Exercise 3.3 ★★ — Lazy word iterator

Implement **`iter_words(text)`** as a **generator** yielding stripped tokens split on whitespace.

Do **not** build the full word list inside the function body before yielding.


In [ ]:
from collections.abc import Iterator


def iter_words(text: str) -> Iterator[str]:
    # YOUR CODE HERE — use yield (generator)
    raise NotImplementedError


assert list(iter_words("  a b
 c")) == ["a", "b", "c"]
assert list(iter_words("")) == []
print("ex 3.3 OK")


### Exercise 3.4 ★★ — `Protocol` for a tiny doc store

Complete **`MemoryStore`** so it satisfies **`DocumentStore`** (implements `get_snippets`).


In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class DocumentStore(Protocol):
    def get_snippets(self, doc_id: str) -> list[str]: ...


class MemoryStore:
    def __init__(self, data: dict[str, list[str]]) -> None:
        self._data = data

    def get_snippets(self, doc_id: str) -> list[str]:
        # YOUR CODE HERE
        raise NotImplementedError


_ds: DocumentStore = MemoryStore({"1": ["chunk-a", "chunk-b"]})
assert isinstance(_ds, DocumentStore)
assert _ds.get_snippets("1") == ["chunk-a", "chunk-b"]
assert _ds.get_snippets("missing") == []
print("ex 3.4 OK")


---
## Part 5 — Solutions (spoilers)

Unfold only **after** attempting Part 4. Running solution cells **overwrites** your experiments — duplicate the notebook first if you care.


In [ ]:
# Exercise 1.1 — solution


def merge_messages(system_prompt: str, user_turns: list[str]) -> list[dict[str, str]]:
    out: list[dict[str, str]] = [{"role": "system", "content": system_prompt}]
    out.extend({"role": "user", "content": u} for u in user_turns)
    return out


msgs = merge_messages("You are concise.", ["Hi", "Summarize yesterday"])
assert msgs[0]["role"] == "system"
print("solution 1.1 OK")


In [ ]:
# Exercise 1.2 — solution
doc_scores = [("doc-a", 0.91), ("doc-b", 0.4), ("doc-c", 0.72)]
threshold = 0.7
filtered_ids = [doc_id for doc_id, score in doc_scores if score >= threshold]
assert filtered_ids == ["doc-a", "doc-c"]
print("solution 1.2 OK")


In [ ]:
# Exercise 1.3 — solution


def prompt_with_constraints(task: str, constraints: dict[str, str]) -> str:
    lines_out = [f"TASK: {task}"]
    for key in sorted(constraints):
        lines_out.append(f"{key}: {constraints[key]}")
    return "
".join(lines_out)


out = prompt_with_constraints("Rewrite email", {"tone": "formal", "length": "short"})
assert out.index("length") < out.index("tone")
print("solution 1.3 OK")


In [ ]:
# Exercise 2.1 — solution


class TextChunker:
    def __init__(self, max_chars: int) -> None:
        self.max_chars = max_chars

    def chunk(self, text: str) -> list[str]:
        if self.max_chars <= 0:
            raise ValueError("max_chars must be positive")
        return [text[i : i + self.max_chars] for i in range(0, len(text), self.max_chars)]


assert TextChunker(5).chunk("abcdefghij") == ["abcde", "fghij"]
print("solution 2.1 OK")


In [ ]:
# Exercise 2.2 — solution
import json
from typing import Any


def safe_parse_json(raw: str) -> dict[str, Any] | None:
    try:
        val = json.loads(raw)
    except json.JSONDecodeError:
        return None
    return val if isinstance(val, dict) else None


assert safe_parse_json("{}") == {}
assert safe_parse_json('"string"') is None
print("solution 2.2 OK")


In [ ]:
# Exercise 2.3 — solution
from pathlib import Path
import json


def count_jsonl_records(path: Path) -> int:
    n = 0
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        try:
            json.loads(line)
        except json.JSONDecodeError:
            continue
        else:
            n += 1
    return n


_tmp = Path("_notebook_tmp_ex")
_tmp.mkdir(exist_ok=True)
_p = _tmp / "sample.jsonl"
_p.write_text('{}

{"x": 1}
', encoding="utf-8")
assert count_jsonl_records(_p) == 2
_p.unlink(missing_ok=True)
_tmp.rmdir()
print("solution 2.3 OK")


In [ ]:
# Exercise 3.1 — solution
from collections.abc import Iterable


def sparse_dot(a: Iterable[float], b: Iterable[float]) -> float:
    aa = list(a)
    bb = list(b)
    if len(aa) != len(bb):
        raise ValueError("length mismatch")
    return sum(x * y for x, y in zip(aa, bb))


assert sparse_dot([1, 2], [3, 4]) == 11
print("solution 3.1 OK")


In [ ]:
# Exercise 3.2 — solution
import functools


def retry_on_value_error(max_attempts: int = 2):
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last_err: ValueError | None = None
            for _ in range(max_attempts):
                try:
                    return fn(*args, **kwargs)
                except ValueError as e:
                    last_err = e
            assert last_err is not None
            raise last_err

        return wrapper

    return deco


_attempts = {"n": 0}


@retry_on_value_error(max_attempts=3)
def _flaky_success():
    _attempts["n"] += 1
    if _attempts["n"] < 3:
        raise ValueError("warmup")
    return "done"


_attempts["n"] = 0
assert _flaky_success() == "done"


@retry_on_value_error(max_attempts=2)
def _flaky_fail():
    raise ValueError("always")


_try_n = {"n": 0}
try:
    _flaky_fail()
except ValueError:
    _try_n["n"] += 1
assert _try_n["n"] == 1

print("solution 3.2 OK")


In [ ]:
# Exercise 3.3 — solution


def iter_words(text: str):
    for token in text.split():
        yield token


assert list(iter_words("  a b
 c")) == ["a", "b", "c"]
print("solution 3.3 OK")


In [ ]:
# Exercise 3.4 — solution
from typing import Protocol, runtime_checkable


@runtime_checkable
class DocumentStore(Protocol):
    def get_snippets(self, doc_id: str) -> list[str]: ...


class MemoryStore:
    def __init__(self, data: dict[str, list[str]]) -> None:
        self._data = data

    def get_snippets(self, doc_id: str) -> list[str]:
        return list(self._data.get(doc_id, []))


_ds: DocumentStore = MemoryStore({"1": ["chunk-a"]})
assert _ds.get_snippets("missing") == []
print("solution 3.4 OK")


---
## Next steps in this repo

- Implement drills from **`CURRICULUM-A-Z.md`** under `exercises/`.
- Move to **`04-api-engineering/`** + **`05-fastapi-projects/`** when HTTP services click.
- Log sessions in **`01-daily-log/`**.

**Suggested libraries later (outside stdlib):** `httpx`, `pydantic`, `numpy`, `pytest` — add them when projects need them.